<a href="https://colab.research.google.com/github/adryann-olivatti/Damicore_IC_notebook1_jose/blob/main/Resumo_1_inicia%C3%A7%C3%A3o_cientifica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

* Resumo doque faz o notebook 1 do josé

* Oque são variáveis nesse notebook:

As Variáveis (ou atributos) são simplesmente as colunas da planilha de dados (Base2FABC.xlsx). Cada linha representa uma lavoura/talhão específico, e cada coluna é uma medição, característica ou fator registrado sobre essa lavoura.

No contexto desse conjunto de dados de 203 colunas, essas variáveis se dividem em:

    
* Variável Alvo (Dependente): É o resultado final que você quer explicar ou otimizar. No código, é a Produtividade [kg/ha].

* Variáveis de Solo e Nutrição: Mapeiam a química e a fertilidade da terra (ex: pH do solo, Teor de Fósforo, Teor de Potássio, Matéria Orgânica, Saturação por Bases).

* Variáveis Climáticas: Registram as condições do tempo ao longo do ciclo (ex: Precipitação acumulada, Temperatura média, Graus Dias Acumulados (GDA), Dias de Estresse Hídrico).

* Variáveis de Manejo Agronômico: Medem as práticas técnicas adotadas pelo produtor (ex: Data de Semeadura, Espaçamento entre linhas, Dose de Adubação (N, P, K), Variedade/Cultivar da semente, População de plantas).

* Variáveis Geográficas e Topográficas: Medem as características físicas da área (ex: Altitude Máxima, Altitude Mínima, Declividade do terreno).

O objetivo do método FS-OPA é pegar cada uma dessas centenas de colunas (variáveis) e descobrir numericamente quais delas têm maior influência direta sobre a coluna da Produtividade.

O notebook implementa o método FS-OPA (Feature Sensitivity through criterion-based resampling from Phylogram Analysis - Otimização/Sensibilidade de Atributos baseada em Análise de Filogramas).

* Visão Geral do Notebook

O objetivo do código é identificar quais variáveis (atributos agronômicos, climáticos, de solo e manejo) são mais determinantes para a Produtividade (kg/ha) de lavouras. Para isso, ele:

1 - Filtra e divide o conjunto de dados nos melhores (Best) e piores (Worst) desempenhos de produtividade em diferentes proporções (50%, 25% e 12,5%).

2 - Visualiza e compara as distribuições desses grupos (Gráficos de Violino, Stripplot e Densidade KDE).

3 - Converte cada variável em um arquivo individual, sanitizando os nomes para processamento.

4 - Aplica a ferramenta DAMICORE (Data Mining by Compression), que usa a Distância de Compressão Normalizada (NCD) para agrupar e construir árvores filogenéticas (filogramas) de similaridade entre as variáveis.

 5 - Carrega e visualiza a árvore filogenética resultante utilizando a biblioteca

--------------------------------------

* BLOCO 1: Carregamento do Dataset e Inspeção Inicial
O que faz:

Conecta o Google Colab ao Google Drive, carrega a planilha Excel com os dados agrícolas (Base2FABC.xlsx) e exibe as primeiras linhas e dimensões da base.

In [ ]:
import pandas as pd
from google.colab import drive

# 1. Monta o Google Drive no ambiente Colab para acessar arquivos
drive.mount('/content/drive/')

# 2. Define o caminho da base de dados no Drive
path_f = '/content/drive/MyDrive/FABC/Base2FABC.xlsx'

# 3. Lê a planilha Excel especificando a aba (sheet_name=1)
df_fabc = pd.read_excel(path_f, sheet_name=1)

# 4. Exibe as 5 primeiras linhas do DataFrame
df_fabc.head()

# 5. Verifica o número de linhas (amostras) e colunas (variáveis)
# Resultado: (1496 amostras, 203 variáveis)
df_fabc.shape

BLOCO 2: Filtragem por Critério (Criação das Categorias B2C, W2C, B4C, W4C, B8C, W8C)O que faz:Cria a função df_quartile para ordenar o dataset pela coluna 'Produtividade [kg/ha]' e extrair as frações correspondentes às melhores e piores amostras:
* 2 Categorias (2C - 50%): $B_{2C}$ (50% melhores) e $W_{2C}$ (50% piores).
* 4 Categorias (4C - 25%): $B_{4C}$ (25% melhores / 1º Quartil) e $W_{4C}$ (25% piores / 4º Quartil).
* 8 Categorias (8C - 12.5%): $B_{8C}$ (12,5% melhores) e $W_{8C}$ (12,5% piores).

In [ ]:
import pandas as pd


# Função para ordenar a base e recortar uma fração (percentual/quartil)
def df_quartile(
    df, criterion, porc=0.25, quartile='first', ascending=False
):
  # Calcula a quantidade de amostras correspondente à porcentagem desejada
  qsize = 1 + int(df[df.columns[0]].count() * porc)

  # Mapeia os intervalos de corte
  dic = {
      'first': (0 * qsize, 1 * qsize - 1),
      'second': (1 * qsize, 2 * qsize - 1),
      'center': (int(1.5 * qsize), int(2.5 * qsize - 1)),
  }

  # Ordena o DataFrame de acordo com o critério (ex: Produtividade)
  dfsor = df.sort_values([criterion], ascending=[ascending])

  # Seleciona as linhas da fatia correspondente
  dfqua = pd.DataFrame(
      dfsor.iloc[dic[quartile][0] : dic[quartile][1]]
  ).copy(deep=True)
  return dfqua


# Criação dos subconjuntos de Melhores (B) e Piores (W)
B2C = df_quartile(
    df_fabc,
    'Produtividade [kg/ha]',
    porc=0.50,
    quartile='first',
    ascending=False,
)  # 50% Melhores
W2C = df_quartile(
    df_fabc,
    'Produtividade [kg/ha]',
    porc=0.50,
    quartile='first',
    ascending=True,
)  # 50% Piores

B4C = df_quartile(
    df_fabc,
    'Produtividade [kg/ha]',
    porc=0.25,
    quartile='first',
    ascending=False,
)  # 25% Melhores
W4C = df_quartile(
    df_fabc,
    'Produtividade [kg/ha]',
    porc=0.25,
    quartile='first',
    ascending=True,
)  # 25% Piores

B8C = df_quartile(
    df_fabc,
    'Produtividade [kg/ha]',
    porc=0.125,
    quartile='first',
    ascending=False,
)  # 12.5% Melhores
W8C = df_quartile(
    df_fabc,
    'Produtividade [kg/ha]',
    porc=0.125,
    quartile='first',
    ascending=True,
)  # 12.5% Piores

# Exibe a quantidade de amostras de cada grupo gerado
print('B2C:', B2C.shape)  # (746, 203)
print('W2C:', W2C.shape)  # (746, 203)
print('B4C:', B4C.shape)  # (372, 203)
print('W4C:', W4C.shape)  # (372, 203)
print('B8C:', B8C.shape)  # (185, 203)
print('W8C:', W8C.shape)  # (185, 203)

* BLOCO 3: Estatísticas Descritivas da Produtividade
O que faz:

Aplica o método .describe() na variável 'Produtividade [kg/ha]' para avaliar média, desvio padrão, mínimo, máximo e quartis da base completa e de cada subconjunto.

In [ ]:
# Avaliação estatística da produtividade no dataset original e nos recortes
df_fabc['Produtividade [kg/ha]'].describe()  # Média geral: ~4117 kg/ha
B2C['Produtividade [kg/ha]'].describe()  # Média dos 50% Melhores: ~4665 kg/ha
W2C['Produtividade [kg/ha]'].describe()  # Média dos 50% Piores: ~3569 kg/ha
B4C['Produtividade [kg/ha]'].describe()  # Média dos 25% Melhores: ~4922 kg/ha
W4C['Produtividade [kg/ha]'].describe()  # Média dos 25% Piores: ~3175 kg/ha
B8C['Produtividade [kg/ha]'].describe()  # Média dos 12.5% Melhores: ~5124 kg/ha
W8C['Produtividade [kg/ha]'].describe()  # Média dos 12.5% Piores: ~2847 kg/ha

* BLOCO 4: Visualização das Distribuições (Violin plot + Stripplot e KDE)
O que faz:

Combina os DataFrames adicionando uma coluna 'Label' e desenha:

    Gráfico de Violino com Pontos (Stripplot): Mostra a dispersão e densidade de pontos da produtividade em cada categoria.

    Gráfico de Densidade (KDE): Compara as curvas de densidade de probabilidade da produtividade entre os grupos.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# 1. Adiciona rótulos para identificar a origem de cada registro
df_fabc['Label'] = 'df_fabc'
B2C['Label'] = 'B2C'
W2C['Label'] = 'W2C'
B4C['Label'] = 'B4C'
W4C['Label'] = 'W4C'
B8C['Label'] = 'B8C'
W8C['Label'] = 'W8C'

# 2. Une todos os subconjuntos em um único DataFrame unificado
dataframes = [df_fabc, B2C, W2C, B4C, W4C, B8C, W8C]
df_combined = pd.concat(dataframes)

# 3. Plota o Gráfico de Violino + Stripplot
plt.figure(figsize=(14, 8))
sns.violinplot(
    x='Label',
    y='Produtividade [kg/ha]',
    data=df_combined,
    palette='viridis',
    inner='point',
    density_norm='width',
)
sns.stripplot(
    x='Label',
    y='Produtividade [kg/ha]',
    data=df_combined,
    color='k',
    alpha=0.6,
    jitter=0.15,
    size=3,
)
plt.title(
    'Distribuição da Produtividade [kg/ha] entre df_fabc e Subconjuntos',
    fontsize=16,
)
plt.ylabel('Produtividade [kg/ha]', fontsize=14)
plt.xlabel('Conjunto de Dados', fontsize=14)
plt.grid(True, which='major', linestyle='--', linewidth=0.5)
plt.tight_layout()
plt.show()

# 4. Plota as Curvas de Densidade KDE
plt.figure(figsize=(18, 6))
for df, label in zip(
    dataframes, ['df_fabc', 'B2C', 'W2C', 'B4C', 'W4C', 'B8C', 'W8C']
):
  sns.kdeplot(
      df['Produtividade [kg/ha]'], label=label, fill=True, alpha=0.5
  )

plt.title('Densidade de Produtividade [kg/ha] entre df_fabc e Subconjuntos')
plt.xlabel('Produtividade [kg/ha]')
plt.ylabel('Densidade')
plt.legend()
plt.show()

* BLOCO 5: Conversão de Variáveis em Arquivos CSV Individuais
O que faz:

O algoritmo DAMICORE trabalha comparando arquivos individuais para calcular a entropia/compressão entre atributos.
Este bloco percorre cada coluna do DataFrame, remove acentos e caracteres especiais dos nomes das variáveis (usando unidecode) e salva cada variável como um arquivo CSV separado no Google Drive (para os 6 subconjuntos: B2C, W2C, B4C, W4C, B8C, W8C).

* Exemplo do subconjunto B2C:

In [ ]:
import os
from unidecode import unidecode

# 1. Define o diretório de destino no Google Drive
b2c_dir = "/content/drive/MyDrive/FABC/Etapa13/data3/B2C"
os.makedirs(b2c_dir, exist_ok=True)

dic = {}
fnamel = []
k = 0
chars2replac = ['_', ' ', '[', ']', '/', '<', '>', '.', '%', '=', '(', ')']

# 2. Itera sobre cada variável (coluna) do subconjunto B2C
for i in B2C.columns:
    fname = str(i)
    # Remove acentos e caracteres não-ASCII
    fname = unidecode(fname)
    # Remove caracteres especiais do nome do arquivo
    for c in chars2replac:
        fname = fname.replace(c, '')

    dic[fname] = str(i) # Mapeamento nome limpo -> nome original

    # 3. Define o caminho completo e grava o CSV da coluna individual
    full_path = os.path.join(b2c_dir, fname)
    fnamel.append(full_path)

    with open(full_path, 'w') as f:
        B2C[i].to_csv(f)
        k += 1

print(f"Verificador de sucesso: {k} arquivos escritos em '{b2c_dir}'.")

* BLOCO 6: Instalação e Preparação do DAMICORE
O que faz:

Instala o repositório da biblioteca DAMICORE via Git, a biblioteca python-igraph e localiza os módulos do pacote na estrutura de pastas do Python no Google Colab.

In [ ]:
import os
import sys

# 1. Instala a biblioteca DAMICORE diretamente do repositório GitLab da USP
!pip install git+https://gitlab.uspdigital.usp.br/jjuarez/damicore.git
!pip install python-igraph

# 2. Localiza o caminho de instalação do pacote DAMICORE
py_version = f"{sys.version_info.major}.{sys.version_info.minor}"
path = f'/usr/local/lib/python{py_version}/dist-packages/damicore/'
os.chdir(path)

# 3. Importa o módulo damicore
import damicore as dm

* BLOCO 7: Execução do DAMICORE para Criação dos Filogramas
O que faz:

Executa o algoritmo damicore.py para cada pasta contendo as variáveis. O DAMICORE:

* Comprime pares de arquivos usando o compressor gzip.

* Calcula a Matriz de Distância NCD (Normalized Compression Distance).

* Constrói a árvore filogenética (arquivo .newick) e detecta comunidades/agrupamentos de variáveis similares.

In [ ]:
import os
import shutil

# --- Exemplo da execução usando cópia local (mais rápida) para B4C ---

# 1. Define diretórios de entrada/saída no Drive e na memória local do Colab
drive_input_dir = "/content/drive/MyDrive/FABC/Etapa13/data3/B4C"
drive_output_dir = "/content/drive/MyDrive/FABC/Etapa13/data3/B4Coutput"
local_input_dir = "/content/B4C_local"
local_output_dir = "/content/B4Coutput_local"

# 2. Copia arquivos para o disco local do Colab para acelerar a compressão
os.makedirs(local_output_dir, exist_ok=True)
if os.path.exists(local_input_dir):
    shutil.rmtree(local_input_dir)
shutil.copytree(drive_input_dir, local_input_dir)

# 3. Define numeração para evitar sobrescrever execuções anteriores
existing_files = os.listdir(drive_output_dir)
max_num = max([int(f.split("-")[0]) for f in existing_files if "-" in f and f.split("-")[0].isdigit()] or [0])
next_num = max_num + 1

# 4. Define argumentos do DAMICORE
argv = [
    "damicore.py",
    local_input_dir,
    "--compressor", "gzip", # Algoritmo de compressão de dados
    "--tree-output", os.path.join(local_output_dir, f"{next_num}-tree.newick"), # Árvore filogenética
    "--output", os.path.join(local_output_dir, f"{next_num}-membership.csv"),
    "--partition-output", os.path.join(local_output_dir, f"{next_num}-partition.csv"),
    "--ncd-output", os.path.join(local_output_dir, f"{next_num}-ncd.csv"), # Matriz de distância
    "--community-detection", "fast" # Algoritmo de detecção de comunidades
]

# 5. Executa a função principal do DAMICORE
dm.main(argv[1:])

# 6. Copia os resultados gerados de volta para o Google Drive
for item in os.listdir(local_output_dir):
    shutil.copy(os.path.join(local_output_dir, item), drive_output_dir)

* BLOCO 8: Carregamento e Análise da Árvore Filogenética Gerada
O que faz:

 Carrega a árvore criada no formato Newick (1-tree.newick), instala a biblioteca de manipulação de árvores ete3 e resolve compatibilidades de versão no Python (com legacy-cgi).

In [ ]:
import sys
from google.colab import files

# 1. Permite carregar manualmente um arquivo .newick gerado
files.upload()

# 2. Instala biblioteca ete3 para renderização e análise de árvores filogenéticas
!pip install ete3

# 3. Corrige compatibilidade do módulo 'cgi' obsoleto no Python 3.13+
!pip install -q legacy-cgi
try:
  import cgi
except ImportError:
  import legacy_cgi as cgi

  sys.modules['cgi'] = cgi

# 4. Importa a classe Tree do ete3
from ete3 import Tree

# 5. Lê e parseia a estrutura hierárquica da árvore filogenética
# (Mostra os relacionamentos de similaridade e distâncias entre as variáveis como 'AltitudeMaxm', 'Produtividade', 'GDA', etc.)

* Resumo dos Resultados Gerados:

    Matriz NCD: Medida estatística baseada em Teoria da Informação de quão similares são duas variáveis.

    Árvore Filogenética (.newick): Mostra quais variáveis ficam agrupadas no mesmo ramo da variável alvo (Produtividade [kg/ha]). As variáveis mais próximas na árvore indicam maior sensibilidade/impacto sobre a produtividade do grupo.

In [ ]:
┌────────────────────────────────────────────────────────┐
  │                 Dataset Agrícola Bruto                 │
  │                  (1496 linhas, 203 cols)               │
  └──────────────────────────┬─────────────────────────────┘
                             │
                             ▼
  ┌────────────────────────────────────────────────────────┐
  │         Fatiamento por Critério (Produtividade)         │
  │          B2C/W2C (50%)  |  B4C/W4C (25%) | B8C/W8C (12.5%)│
  └──────────────────────────┬─────────────────────────────┘
                             │
                             ▼
  ┌────────────────────────────────────────────────────────┐
  │      Sanitização de Nomes & Geração de CSVs Indiv.     │
  │        (unidecode, remoção de caracteres de pontuação) │
  └──────────────────────────┬─────────────────────────────┘
                             │
                             ▼
  ┌────────────────────────────────────────────────────────┐
  │                  Algoritmo DAMICORE                    │
  │  - Compressão GZIP de pares de arquivos                │
  │  - Matriz NCD (Normalized Compression Distance)         │
  │  - Agrupamento Neighbor-Joining                        │
  └──────────────────────────┬─────────────────────────────┘
                             │
                             ▼
  ┌────────────────────────────────────────────────────────┐
  │            Árvore Filogenética (.newick)               │
  │   - Medição da Distância Cofenética até Produtividade  │
  │   - Ranking final das Variáveis mais Relevantes/Sensíveis│
  └────────────────────────────────────────────────────────┘